In [1]:
import pandas as pd
from pathlib import Path

In [2]:
transients = pd.read_parquet(
    "snia_results.parquet",
    engine="fastparquet",
)

In [3]:
transients

,diaObjectId,tns_name,z,catalogue_object_id,tns_prefix,type,ra,decl,absMag,nDiaSources
0,170019717277810735,2026ctw,0.150000,"10000,342494,11570",SN,SN Ia,151.740293,1.927064,NaN,754
1,170028486191218743,2026qrh,0.100000,"10000,81273,866",SN,SN Ia,64.185342,-48.943229,NaN,88
2,170028500610711681,2024xom,0.044167,CGCG036-024,SN,SN Ia,150.746125,2.336469,NaN,165
3,170028510695916335,2025mbp,0.046160,2MASXJ10062208+0109317,SN,SN Ia,151.592617,1.158385,NaN,273
4,170028520834072659,2026fgl,0.164000,WISEAJ095652.35+032811.7,SN,SN Ia,149.218165,3.469919,-16.5385,19
...,...,...,...,...,...,...,...,...,...,...
122,170661828398940941,2025bbx,0.067000,14470795-3930390,SN,SN Ia,221.785403,-39.509297,NaN,3
123,313893021499785278,2026ezw,0.131000,SDSSJ095832.59+011610.2,SN,SN Ia,149.635687,1.269344,NaN,20
124,313932566867476549,2026qrh,0.100000,"10000,81273,866",SN,SN Ia,64.185342,-48.943229,NaN,9
125,313972182793322538,2026fgl,0.164000,WISEAJ095652.35+032811.7,SN,SN Ia,149.218165,3.469919,-16.2020,17


In [4]:
transients = (
    transients[["diaObjectId", "tns_name", "ra", "decl"]]
    .rename(columns={"decl": "dec"})
    .dropna()
    .reset_index(drop=True)
    .copy()
)

transients["tns_name"] = transients["tns_name"].astype(str)

print(f"Associating {len(transients)} transients")
display(transients)

Associating 127 transients


,diaObjectId,tns_name,ra,dec
0,170019717277810735,2026ctw,151.740293,1.927064
1,170028486191218743,2026qrh,64.185342,-48.943229
2,170028500610711681,2024xom,150.746125,2.336469
3,170028510695916335,2025mbp,151.592617,1.158385
4,170028520834072659,2026fgl,149.218165,3.469919
...,...,...,...,...
122,170661828398940941,2025bbx,221.785403,-39.509297
123,313893021499785278,2026ezw,149.635687,1.269344
124,313932566867476549,2026qrh,64.185342,-48.943229
125,313972182793322538,2026fgl,149.218165,3.469919


In [ ]:
import inspect
import textwrap
import traceback

import pandas as pd
import astro_prost.helpers as prost_helpers
from scipy.stats import gamma, uniform
from astro_prost import associate_sample

# Temporary read-only fix without editing helpers.py.
if not getattr(prost_helpers, "_shape_r_copy_patch_applied", False):
    source = textwrap.dedent(
        inspect.getsource(prost_helpers.calc_shape_props_decals)
    )

    old = 'candidate_hosts["shape_r"].values'
    new = 'candidate_hosts["shape_r"].to_numpy(copy=True)'

    if old not in source:
        raise RuntimeError("Expected shape_r code not found.")

    source = source.replace(old, new)
    exec(
        compile(source, "<prost_notebook_patch>", "exec"),
        prost_helpers.__dict__,
    )
    prost_helpers._shape_r_copy_patch_applied = True

# Collect results in memory.
results = []
failed = []

for i in range(len(transients)):
    one_transient = transients.iloc[[i]].copy().reset_index(drop=True)
    name = str(one_transient.iloc[0]["tns_name"])

    try:
        result = associate_sample(
        one_transient,
        catalogs=["decals", "panstarrs", "skymapper"],
        name_col="tns_name",
        coord_cols=("ra", "dec"),
        priors={"offset": uniform(loc=0, scale=10)},
        likes={"offset": gamma(a=0.75)},
        n_samples=1000,  # Match the 1,000 samples used internally.
        parallel=False,
        save=False,
        )
    

        if isinstance(result, Exception):
            raise result

        if not isinstance(result, pd.DataFrame):
            raise TypeError(
                f"Expected DataFrame; got {type(result).__name__}"
            )

        if result.empty:
            raise ValueError("PROST returned an empty DataFrame.")

        results.append(result)
        print(f"{i + 1}/{len(transients)} done: {name}")

    except Exception as exc:
        failed.append({"tns_name": name, "error": repr(exc)})
        print(f"{i + 1}/{len(transients)} failed: {name}: {exc}")
        traceback.print_exc()

hosts = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
failed_df = pd.DataFrame(failed, columns=["tns_name", "error"])

display(hosts)
display(failed_df)

2026-10-14 11:11:42,749- INFO - Conditioning association on the following properties: ['offset']
2026-10-14 11:11:44,913- INFO - Loaded GLADE+ catalog.
2026-10-14 11:11:44,916- INFO - 

Associating 2026ctw at RA, DEC = 151.740293, 1.927064 (redshift nan)
2026-10-14 11:11:47,741- INFO - Association successful!
2026-10-14 11:11:47,760- INFO - 
    
Properties of best host (in decals dr9) for 2026ctw:
    objID: 9906622211952776
    R.A. (deg): 151.740166
    Dec. (deg): 1.927030
    Posterior: 1.0000e+00
    Redshift: 0.7213 ± 0.3376 (PHOT)
    Absolute Magnitude: -20.3 ± 1.1 (r)
    Transient Offset: 0.5 ± 0.1 arcsec
    Transient Offset Posterior: 1.0000e+00
2026-10-14 11:11:47,760- INFO - 
    
Properties of 2nd best host (in decals dr9) for 2026ctw:
    Redshift: 1.0042 ± 0.1648 (PHOT)
    Absolute Magnitude: -19.8 ± 1.2 (r)
    Transient Offset: 292.4 ± 0.1 arcsec
    Transient Offset Posterior: 0.0000e+00
2026-10-14 11:11:47,761- INFO - Chosen galaxy has catalog ID of 9906622211952

1/127 done: 2026ctw


2026-10-14 11:11:49,748- INFO - Loaded GLADE+ catalog.
2026-10-14 11:11:49,750- INFO - 

Associating 2026qrh at RA, DEC = 64.185342, -48.943229 (redshift nan)
2026-10-14 11:11:52,642- INFO - Association successful!
2026-10-14 11:11:52,685- INFO - 
    
Properties of best host (in decals dr9) for 2026qrh:
    objID: 9906605092569380
    R.A. (deg): 64.185370
    Dec. (deg): -48.943233
    Posterior: 9.8797e-01
    Redshift: 0.1125 ± 0.0167 (PHOT)
    Absolute Magnitude: -20.9 ± 0.9 (r)
    Transient Offset: 0.1 ± 0.1 arcsec
    Transient Offset Posterior: 9.8797e-01
2026-10-14 11:11:52,686- INFO - 
    
Properties of 2nd best host (in decals dr9) for 2026qrh:
    Redshift: 1.2030 ± 0.1423 (PHOT)
    Absolute Magnitude: -21.5 ± 1.2 (r)
    Transient Offset: 297.7 ± 0.1 arcsec
    Transient Offset Posterior: 0.0000e+00
2026-10-14 11:11:52,686- INFO - Chosen galaxy has catalog ID of 9906605092569380 and RA, DEC = 64.185370, -48.943233
2026-10-14 11:11:52,697- INFO - Conditioning associatio

2/127 done: 2026qrh


2026-10-14 11:11:54,657- INFO - Loaded GLADE+ catalog.
2026-10-14 11:11:54,659- INFO - 

Associating 2024xom at RA, DEC = 150.746125, 2.336469 (redshift nan)
2026-10-14 11:11:56,601- INFO - Association successful!
2026-10-14 11:11:56,626- INFO - 
    
Properties of best host (in decals dr9) for 2024xom:
    objID: 9906622306063947
    R.A. (deg): 150.743722
    Dec. (deg): 2.335828
    Posterior: 7.3390e-01
    Redshift: 0.1483 ± 0.0741 (PHOT)
    Absolute Magnitude: -20.1 ± 1.0 (r)
    Transient Offset: 8.9 ± 0.1 arcsec
    Transient Offset Posterior: 7.3390e-01
2026-10-14 11:11:56,627- INFO - 
    
Properties of 2nd best host (in decals dr9) for 2024xom:
    Redshift: 0.7043 ± 0.3117 (PHOT)
    Absolute Magnitude: -19.5 ± 1.2 (r)
    Transient Offset: 299.4 ± 0.1 arcsec
    Transient Offset Posterior: 0.0000e+00
2026-10-14 11:11:56,627- INFO - Chosen galaxy has catalog ID of 9906622306063947 and RA, DEC = 150.743722, 2.335828
2026-10-14 11:11:56,639- INFO - Conditioning association o

...

2026-10-14 11:14:31,482- INFO - Loaded GLADE+ catalog.
2026-10-14 11:14:31,483- INFO - 

Associating 2026ftl at RA, DEC = 188.402327, 6.368380 (redshift nan)
2026-10-14 11:14:33,421- INFO - Association successful!
2026-10-14 11:14:33,443- INFO - 
    
Properties of best host (in decals dr9) for 2026ftl:
    objID: 9906623822760158
    R.A. (deg): 188.402259
    Dec. (deg): 6.368652
    Posterior: 1.0000e+00
    Redshift: 0.3447 ± 0.1104 (PHOT)
    Absolute Magnitude: -20.0 ± 1.1 (r)
    Transient Offset: 1.0 ± 0.1 arcsec
    Transient Offset Posterior: 1.0000e+00
2026-10-14 11:14:33,444- INFO - 
    
Properties of 2nd best host (in decals dr9) for 2026ftl:
    objID: 9906623822759252
    R.A. (deg): 188.356962
    Dec. (deg): 6.368931
    Posterior: 0.0000e+00
    Redshift: 0.7597 ± 0.1346 (PHOT)
    Absolute Magnitude: -20.4 ± 1.1 (r)
    Transient Offset: 162.3 ± 0.1 arcsec
    Transient Offset Posterior: 0.0000e+00
2026-10-14 11:14:33,444- INFO - Chosen galaxy has catalog ID of 9906

40/127 done: 2026ftl


2026-10-14 11:14:35,767- INFO - Loaded GLADE+ catalog.
2026-10-14 11:14:35,788- INFO - 

Associating 2026ivl at RA, DEC = 148.474283, 1.718233 (redshift nan)
2026-10-14 11:14:38,623- INFO - Association successful!
2026-10-14 11:14:38,654- INFO - 
    
Properties of best host (in decals dr9) for 2026ivl:
    objID: 9906622116729406
    R.A. (deg): 148.474148
    Dec. (deg): 1.719041
    Posterior: 6.3173e-01
    Redshift: 0.1035 ± 0.0247 (PHOT)
    Absolute Magnitude: -19.3 ± 1.0 (r)
    Transient Offset: 3.0 ± 0.1 arcsec
    Transient Offset Posterior: 6.3173e-01
2026-10-14 11:14:38,654- INFO - 
    
Properties of 2nd best host (in decals dr9) for 2026ivl:
    Redshift: 1.0438 ± 0.2868 (PHOT)
    Absolute Magnitude: -20.8 ± 1.2 (r)
    Transient Offset: 299.0 ± 0.1 arcsec
    Transient Offset Posterior: 0.0000e+00
2026-10-14 11:14:38,655- INFO - Chosen galaxy has catalog ID of 9906622116729406 and RA, DEC = 148.474148, 1.719041
2026-10-14 11:14:38,695- INFO - Conditioning association o

41/127 done: 2026ivl


2026-10-14 11:14:42,795- INFO - Loaded GLADE+ catalog.
2026-10-14 11:14:42,800- INFO - 

Associating 2026kfw at RA, DEC = 149.219885, 1.833104 (redshift nan)
2026-10-14 11:14:45,030- WARNING - Association failed. Host is likely outside the search cone.
2026-10-14 11:14:45,063- INFO - Running with shred_cut = True...


In [ ]:
results